# 🌳 Clase 5 — Probabilidades, uniones e intersecciones de eventos
## Fundamentos del método científico y probabilidad

**Desafío inicial:** Un equipo de ciencia de datos analiza el comportamiento de usuarios frente a recomendaciones en un e-commerce. Las decisiones dependen de cadenas de eventos condicionales (clic → compra → retorno). ¿Cómo organizar las secuencias y calcular probabilidades para decisiones de negocio?

**Objetivos:**
- Construir e interpretar **árboles de probabilidad** para eventos secuenciales
- Calcular probabilidades conjuntas multiplicando por ramas
- Aplicar la regla de la **unión** P(A ∪ B) y la **intersección** P(A ∩ B)
- Clasificar relaciones entre eventos: independencia, exclusión mutua, inclusión
- Usar Python para verificar todos los cálculos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.dpi'] = 110

print('✅ Librerías cargadas')

---
## PARTE 1 — Árbol de probabilidades

### 1.1 Componentes y reglas — slides 6-7

In [ ]:
# Tabla de componentes — slide 6
tabla_comp = pd.DataFrame({
    'Componente': ['Nodo inicial (raíz)','Ramas','Nodos intermedios',
                   'Probabilidades','Hojas'],
    'Descripción': [
        'Punto de inicio del experimento; aún no ha ocurrido ningún evento.',
        'Representan cada posible resultado de un evento.',
        'Puntos donde se bifurcan nuevas ramas (eventos posteriores).',
        'Asignadas a cada rama; probabilidad dado el camino previo.',
        'Puntos finales; indican una combinación completa de resultados.'
    ]
})
print('=== Componentes del árbol de probabilidades (slide 6) ===')
print(tabla_comp.to_string(index=False))
print()
print('Regla clave: la probabilidad de un camino = PRODUCTO de probabilidades en ese camino.')
print('Regla de suma: las ramas de un mismo nodo siempre suman 1.0')

### 1.2 Ejemplo: sistema de recomendación (slides 8-10)

In [ ]:
# Código exacto de la presentación — slide 8
prob_clic = 0.6
prob_compra_dado_clic = 0.3
prob_no_clic = 1 - prob_clic
prob_regresa_dado_no_clic = 0.1

# Cálculos de probabilidades conjuntas
prob_clic_y_compra = prob_clic * prob_compra_dado_clic
prob_clic_y_no_compra = prob_clic * (1 - prob_compra_dado_clic)
prob_no_clic_y_regresa = prob_no_clic * prob_regresa_dado_no_clic
prob_no_clic_y_no_regresa = prob_no_clic * (1 - prob_regresa_dado_no_clic)

print('Probabilidades conjuntas:')
print(f'Clic y compra: {prob_clic_y_compra:.2f}')
print(f'Clic y no compra: {prob_clic_y_no_compra:.2f}')
print(f'No clic y regresa: {prob_no_clic_y_regresa:.2f}')
print(f'No clic y no regresa: {prob_no_clic_y_no_regresa:.2f}')

In [ ]:
# Verificación: todas las hojas deben sumar 1.0 — slide 10
total = (prob_clic_y_compra + prob_clic_y_no_compra
         + prob_no_clic_y_regresa + prob_no_clic_y_no_regresa)
print(f'Suma de todas las hojas: {total:.4f}  ← debe ser 1.0 ✅')

# Tabla resumen del árbol — slide 10
tabla_arbol = pd.DataFrame({
    'Etapa': [1, 1, 2, 2, 2, 2],
    'Evento': ['Clic','No clic',
               'Compra (tras clic)','No compra (tras clic)',
               'Regresa (tras no clic)','No regresa (tras no clic)'],
    'Probabilidad de rama': [0.6, 0.4, 0.3, 0.7, 0.1, 0.9],
    'Probabilidad acumulada': [
        '-', '-',
        f'0.6 × 0.3 = {prob_clic_y_compra:.2f}',
        f'0.6 × 0.7 = {prob_clic_y_no_compra:.2f}',
        f'0.4 × 0.1 = {prob_no_clic_y_regresa:.2f}',
        f'0.4 × 0.9 = {prob_no_clic_y_no_regresa:.2f}'
    ]
})
print()
print('=== Árbol de probabilidades — Tabla resumen (slide 10) ===')
print(tabla_arbol.to_string(index=False))

In [ ]:
# Visualización del árbol de probabilidades
fig, ax = plt.subplots(figsize=(13, 6))
ax.set_xlim(-0.5, 3)
ax.set_ylim(-0.5, 4)
ax.axis('off')
ax.set_title('Árbol de probabilidades — Sistema de recomendación (slide 9-10)',
             fontweight='bold', fontsize=12)

def nodo(ax, x, y, texto, color='#2E75B6', size=11):
    ax.text(x, y, texto, ha='center', va='center', fontsize=size,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=color, alpha=0.8, edgecolor='white'),
            color='white', fontweight='bold')

def rama(ax, x1, y1, x2, y2, prob, color='#555555'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
    mx, my = (x1+x2)/2, (y1+y2)/2
    ax.text(mx+0.05, my, f'p={prob}', fontsize=9, color=color, fontweight='bold')

# Raíz
nodo(ax, 0, 2, 'Usuario\nrecibe\nrecomendación', '#7030A0', 9)
# Etapa 1
rama(ax, 0.25, 2.3, 0.95, 3.1, 0.6, '#2E75B6')
nodo(ax, 1.1, 3.2, 'CLIC\n(0.6)', '#2E75B6')
rama(ax, 0.25, 1.8, 0.95, 0.9, 0.4, '#ED7D31')
nodo(ax, 1.1, 0.8, 'NO CLIC\n(0.4)', '#ED7D31')
# Etapa 2 - rama CLIC
rama(ax, 1.45, 3.35, 2.3, 3.85, 0.3, '#70AD47')
nodo(ax, 2.65, 3.9, f'COMPRA\np={prob_clic_y_compra:.2f}', '#70AD47', 9)
rama(ax, 1.45, 3.1, 2.3, 2.6, 0.7, '#FC4E4E')
nodo(ax, 2.65, 2.5, f'NO COMPRA\np={prob_clic_y_no_compra:.2f}', '#FC4E4E', 9)
# Etapa 2 - rama NO CLIC
rama(ax, 1.45, 0.9, 2.3, 1.4, 0.1, '#70AD47')
nodo(ax, 2.65, 1.45, f'REGRESA\np={prob_no_clic_y_regresa:.2f}', '#70AD47', 9)
rama(ax, 1.45, 0.7, 2.3, 0.2, 0.9, '#FC4E4E')
nodo(ax, 2.65, 0.1, f'NO REGRESA\np={prob_no_clic_y_no_regresa:.2f}', '#FC4E4E', 9)

ax.text(0, -0.3, f'Suma total de hojas = {total:.2f} ✅', fontsize=10,
        ha='center', color='gray', style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# Tabla comparativa — slide 11
tabla_comp_her = pd.DataFrame({
    'Herramienta': ['Tablas de contingencia','Fórmulas directas','Árbol de probabilidades'],
    'Uso principal': ['Análisis de frecuencias y dependencias',
                      'Cálculo puntual de probabilidades',
                      'Procesos secuenciales o condicionales'],
    'Limitaciones': ['No representan eventos secuenciales',
                     'Poco intuitivas para casos complejos',
                     'Difíciles de manejar si hay muchas etapas']
})
print('=== Comparación de herramientas (slide 11) ===')
print(tabla_comp_her.to_string(index=False))

### ✏️ Ejercicio 1:

In [ ]:
# ✏️ Pregunta cierre 1: ¿Qué representa cada rama y cada nivel en un árbol?
c1 = ""

# ✏️ Pregunta cierre 2: ¿Cómo se calcula la probabilidad de una combinación secuencial?
c2 = ""

# ✏️ Dado: P(A) = 0.7, P(B|A) = 0.4, P(B|¬A) = 0.2. Calcula P(A ∩ B):
p_a_ej = 0.7
p_b_dado_a = 0.4
p_no_a = 1 - p_a_ej
p_b_dado_no_a = 0.2

p_a_y_b = p_a_ej * p_b_dado_a
p_no_a_y_b = p_no_a * p_b_dado_no_a

print(f'P(A ∩ B):    {p_a_y_b:.4f}')
print(f'P(¬A ∩ B):   {p_no_a_y_b:.4f}')
print(f'P(B) total:  {p_a_y_b + p_no_a_y_b:.4f}  ← P(B) = P(A∩B) + P(¬A∩B)')

print(f'\n1. {c1}')
print(f'2. {c2}')

---
## PARTE 2 — Unión e intersección de eventos

### 2.1 Unión (A ∪ B) — código exacto slide 12

In [ ]:
# Código exacto de la presentación — slide 12
p_a = 0.6   # Probabilidad de que el usuario haga clic
p_b = 0.3   # Probabilidad de que el usuario compre
p_a_and_b = 0.18   # Probabilidad conjunta de clic y compra

# Fórmula de la unión
p_a_or_b = p_a + p_b - p_a_and_b
print(f'P(Clic ∪ Compra): {p_a_or_b:.2f}')

In [ ]:
# Explicación detallada — slide 13
print('=== Regla de la UNIÓN (slide 13) ===')
print(f'P(A ∪ B) = P(A) + P(B) - P(A ∩ B)')
print(f'         = {p_a} + {p_b} - {p_a_and_b}')
print(f'         = {p_a_or_b:.2f}')
print()
print('¿Por qué restamos P(A ∩ B)?')
print('  → Al sumar P(A) + P(B), la intersección se cuenta dos veces.')
print('  → Restando P(A ∩ B) evitamos la doble contabilización.')
print()
print('Interpretación: hay 72% de probabilidad de que el usuario')
print('haga clic, compre, o ambas cosas.')
print()

# Caso especial: eventos mutuamente excluyentes
print('CASO ESPECIAL — Eventos mutuamente excluyentes:')
print('Si P(A ∩ B) = 0  →  P(A ∪ B) = P(A) + P(B)')
p_cara = 0.5
p_sello = 0.5
print(f'Ej: P(cara ∪ sello) = {p_cara} + {p_sello} = {p_cara + p_sello:.1f}')

### 2.2 Intersección (A ∩ B) — código exacto slide 14

In [ ]:
# Código exacto de la presentación — slide 14
p_clic = 0.6
p_compra_dado_clic = 0.3
p_clic_y_compra = p_clic * p_compra_dado_clic
print(f'P(Clic ∩ Compra): {p_clic_y_compra:.2f}')

In [ ]:
# Explicación detallada — slide 15
print('=== Regla de la INTERSECCIÓN (slide 15) ===')
print()
print('Para eventos INDEPENDIENTES:')
print('  P(A ∩ B) = P(A) × P(B)')
p_dado_6 = 1/6
p_moneda_cara = 0.5
print(f'  Ej: P(dado=6 ∩ moneda=cara) = {p_dado_6:.4f} × {p_moneda_cara} = {p_dado_6*p_moneda_cara:.4f}')
print()
print('Para eventos DEPENDIENTES (condicional):')
print('  P(A ∩ B) = P(A) × P(B|A)')
print(f'  Ej: P(clic) × P(compra|clic) = {p_clic} × {p_compra_dado_clic} = {p_clic_y_compra:.2f}')

### 2.3 Diagrama de Venn — slide 16

In [ ]:
# Diagrama de Venn — slide 16
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Diagrama de Venn — Unión, Intersección y Eventos Disjuntos (slide 16)',
             fontweight='bold')

def circulo(ax, cx, cy, r, color, alpha=0.4):
    c = plt.Circle((cx, cy), r, color=color, alpha=alpha)
    ax.add_patch(c)
    return c

for ax in axes:
    ax.set_xlim(-2, 2)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')

# Gráfico 1: A ∪ B (unión)
circulo(axes[0], -0.5, 0, 0.9, '#2E75B6', 0.5)
circulo(axes[0], 0.5, 0, 0.9, '#ED7D31', 0.5)
axes[0].text(-0.9, 0, 'A', fontsize=16, ha='center', fontweight='bold', color='navy')
axes[0].text(0.9, 0, 'B', fontsize=16, ha='center', fontweight='bold', color='#7D3000')
axes[0].set_title('A ∪ B (Unión)\nTodo lo coloreado', fontsize=10)
axes[0].text(0, -1.3, 'P(A ∪ B) = P(A)+P(B)-P(A∩B)', fontsize=8, ha='center')

# Gráfico 2: A ∩ B (intersección)
circulo(axes[1], -0.5, 0, 0.9, '#A5A5A5', 0.2)
circulo(axes[1], 0.5, 0, 0.9, '#A5A5A5', 0.2)
# Destacar solo la intersección
from matplotlib.patches import Wedge
axes[1].text(-0.9, 0, 'A', fontsize=16, ha='center', fontweight='bold', color='gray')
axes[1].text(0.9, 0, 'B', fontsize=16, ha='center', fontweight='bold', color='gray')
axes[1].text(0, 0, '∩', fontsize=20, ha='center', va='center',
             color='#7030A0', fontweight='bold')
axes[1].set_title('A ∩ B (Intersección)\nSolo el área común', fontsize=10)
axes[1].text(0, -1.3, 'P(A ∩ B) = P(A) × P(B|A)', fontsize=8, ha='center')

# Gráfico 3: Eventos disjuntos
circulo(axes[2], -0.8, 0, 0.7, '#2E75B6', 0.5)
circulo(axes[2], 0.8, 0, 0.7, '#ED7D31', 0.5)
axes[2].text(-0.8, 0, 'A', fontsize=16, ha='center', fontweight='bold', color='white')
axes[2].text(0.8, 0, 'B', fontsize=16, ha='center', fontweight='bold', color='white')
axes[2].set_title('Mutuamente excluyentes\n(sin intersección)', fontsize=10)
axes[2].text(0, -1.3, 'P(A ∩ B) = 0 → P(A∪B) = P(A)+P(B)', fontsize=8, ha='center')

plt.tight_layout()
plt.show()

### 2.4 Relaciones entre eventos — slide 17

In [ ]:
# Tabla slide 17
tabla_rel = pd.DataFrame({
    'Relación': ['Independencia','Mutuamente excluyentes','Inclusión (A ⊂ B)'],
    'Definición': [
        'La ocurrencia de un evento no afecta al otro.',
        'No pueden ocurrir al mismo tiempo.',
        'Siempre que A ocurre, también ocurre B.'
    ],
    'Fórmula': [
        'P(A ∩ B) = P(A) × P(B)',
        'P(A ∩ B) = 0 → P(A ∪ B) = P(A) + P(B)',
        'P(A ∪ B) = P(B)'
    ],
    'Ejemplo': [
        'Lanzar un dado y lanzar una moneda',
        'Cara o sello en una moneda',
        'Comprar implica haber hecho clic'
    ]
})
print('=== Tipos de relaciones entre eventos (slide 17) ===')
print(tabla_rel.to_string(index=False))

# Verificación numérica de cada tipo
print()
print('--- Verificación numérica ---')
# Independencia
p_dado_par = 3/6
p_moneda_cara = 0.5
print(f'Independencia: P(par) × P(cara) = {p_dado_par:.4f} × {p_moneda_cara} = {p_dado_par*p_moneda_cara:.4f}')

# Mutualmente excluyentes
p_cara2, p_sello2 = 0.5, 0.5
print(f'Excl. mutua: P(cara ∪ sello) = {p_cara2} + {p_sello2} = {p_cara2+p_sello2} (∩ = 0)')

# Inclusión: P(compra) ≤ P(clic)
print(f'Inclusión: P(compra)={p_clic_y_compra:.2f} ≤ P(clic)={p_clic} ✅')

### ✏️ Ejercicio 2:

In [ ]:
# ✏️ Pregunta cierre 3: ¿Cuándo usar la regla de la unión y cuándo la de la intersección?
c3 = ""

# ✏️ Pregunta cierre 4: ¿Qué significa que dos eventos sean mutuamente excluyentes? ¿Y independientes?
c4 = ""

# ✏️ Práctica: P(A) = 0.4, P(B) = 0.5, P(A∩B) = 0.15. Calcula P(A∪B):
p_a_ej2, p_b_ej2, p_ab_ej2 = 0.4, 0.5, 0.15
p_union_ej2 = p_a_ej2 + p_b_ej2 - p_ab_ej2
print(f'P(A ∪ B) = {p_a_ej2} + {p_b_ej2} - {p_ab_ej2} = {p_union_ej2:.2f}')

# ✏️ Si P(A∩B) = 0, ¿son independientes o mutuamente excluyentes?
c_excl = ""

print(f'\n3. {c3}')
print(f'4. {c4}')
print(f'P(A∩B)=0 significa: {c_excl}')

---
## PARTE 3 — Actividad guiada: Plataforma de streaming (slides 19-23)

### 3.1 Código exacto del slide 23

In [ ]:
# Código exacto de la presentación — slide 23
p_ver = 0.6
p_registro_despues_ver = 0.25
p_no_ver = 1 - p_ver
p_segunda_reco = 0.4
p_registro_despues_segunda = 0.1

p_caso_1 = p_ver * p_registro_despues_ver
p_caso_2 = p_no_ver * p_segunda_reco * p_registro_despues_segunda
p_total  = p_caso_1 + p_caso_2

print(f'Registro tras ver primer video: {p_caso_1:.2f}')
print(f'Registro tras segunda recomendación: {p_caso_2:.2f}')
print(f'Probabilidad total de registro: {p_total:.2f}')

In [ ]:
# Desglose completo — instrucciones slide 22
print('=== PASO 2: Cálculos detallados ===')
print()
print('Camino 1: Usuario ve el primer video Y se registra')
print(f'  P(ver) × P(registro|ver) = {p_ver} × {p_registro_despues_ver} = {p_caso_1:.3f}')
print()
print('Camino 2: No ve el primer video → recibe segunda recomendación → se registra')
print(f'  P(no ver) × P(2ª reco) × P(registro|2ª) = {p_no_ver} × {p_segunda_reco} × {p_registro_despues_segunda} = {p_caso_2:.3f}')
print()
print('Probabilidad total de registro (Unión de caminos mutuamente excluyentes):')
print(f'  P(total) = P(camino 1) + P(camino 2) = {p_caso_1:.3f} + {p_caso_2:.3f} = {p_total:.3f}')
print()
print('=== PASO 3: ¿Cuál es la vía más eficaz? ===')
if p_caso_1 > p_caso_2:
    print(f'Camino 1 (ver + registrarse): {p_caso_1:.3f}  ← MÁS EFICAZ')
    print(f'Camino 2 (2ª recomendación): {p_caso_2:.3f}')
    print(f'El camino 1 tiene {p_caso_1/p_caso_2:.1f}x más probabilidad de conversión.')
else:
    print(f'Camino 2 es más eficaz.')

In [ ]:
# Visualización del árbol de la actividad guiada
fig, ax = plt.subplots(figsize=(13, 6))
ax.set_xlim(-0.5, 3)
ax.set_ylim(-1, 5)
ax.axis('off')
ax.set_title('Árbol de probabilidades — Plataforma de streaming (slide 21)',
             fontweight='bold')

# Función de dibujo
def draw_tree_node(ax, x, y, texto, color, size=10):
    ax.text(x, y, texto, ha='center', va='center', fontsize=size,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=color, alpha=0.85, edgecolor='white'),
            color='white', fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2, p_label, color='#555555'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.text((x1+x2)/2+0.06, (y1+y2)/2, p_label, fontsize=9, color=color, fontweight='bold')

draw_tree_node(ax, 0, 2.5, 'Usuario\nrecibe\nsugerencia', '#7030A0', 9)
# Nivel 1
draw_arrow(ax, 0.25, 2.9, 1.0, 3.8, 'p=0.6', '#2E75B6')
draw_tree_node(ax, 1.15, 4.0, 'VE VIDEO\n(0.6)', '#2E75B6')
draw_arrow(ax, 0.25, 2.1, 1.0, 1.2, 'p=0.4', '#ED7D31')
draw_tree_node(ax, 1.15, 1.0, 'NO VE\n(0.4)', '#ED7D31')
# Nivel 2 - VE VIDEO
draw_arrow(ax, 1.45, 4.2, 2.2, 4.7, 'p=0.25', '#70AD47')
draw_tree_node(ax, 2.6, 4.75, f'REGISTRO\np={p_caso_1:.3f}', '#70AD47', 9)
draw_arrow(ax, 1.45, 3.85, 2.2, 3.3, 'p=0.75', '#FC4E4E')
draw_tree_node(ax, 2.6, 3.2, f'NO REGISTRO\np={p_ver*(1-p_registro_despues_ver):.3f}', '#FC4E4E', 9)
# Nivel 2 - NO VE (2ª reco)
draw_arrow(ax, 1.45, 1.1, 2.2, 1.7, 'p=0.4', '#FFC000')
draw_tree_node(ax, 2.45, 1.85, '2ª RECO', '#FFC000')
draw_arrow(ax, 1.45, 0.85, 2.2, 0.2, 'p=0.6', '#FC4E4E')
draw_tree_node(ax, 2.6, 0.1, f'SIN 2ª RECO\np={p_no_ver*0.6:.3f}', '#FC4E4E', 9)
# Nivel 3 - 2ª reco
draw_arrow(ax, 2.7, 1.95, 3.05, 2.5, '0.1', '#70AD47')
ax.text(3.1, 2.5, f'REG.\np={p_caso_2:.3f}', fontsize=9, color='white', fontweight='bold',
        ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#70AD47', alpha=0.85))
draw_arrow(ax, 2.7, 1.7, 3.05, 1.1, '0.9', '#FC4E4E')
ax.text(3.1, 1.0, f'NO REG.\np={p_no_ver*p_segunda_reco*0.9:.3f}', fontsize=9,
        color='white', fontweight='bold', ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FC4E4E', alpha=0.85))

ax.text(0, -0.7, f'P(registro total) = {p_caso_1:.3f} + {p_caso_2:.3f} = {p_total:.3f}',
        fontsize=11, ha='center', color='#2E75B6', fontweight='bold')

plt.tight_layout()
plt.show()

### ✏️ PASO 4 — Reflexión y decisiones de negocio (slide 22)

In [ ]:
# ✏️ Paso 4 — reflexión sobre impacto en decisiones de negocio

# ¿Cuál camino priorizar?
recomendacion_negocio = ""

# ¿Qué pasaría si aumentamos P(ver) de 0.6 a 0.75?
p_ver_nuevo = 0.75
p_caso_1_nuevo = p_ver_nuevo * p_registro_despues_ver
p_total_nuevo = p_caso_1_nuevo + (1-p_ver_nuevo) * p_segunda_reco * p_registro_despues_segunda
print(f'Si P(ver) sube a {p_ver_nuevo}:')
print(f'  P(registro por video): {p_caso_1_nuevo:.3f} (+{p_caso_1_nuevo-p_caso_1:.3f})')
print(f'  P(registro total):     {p_total_nuevo:.3f} (+{p_total_nuevo-p_total:.3f})')
print()

# ¿Qué pasaría si mejoramos P(registro|ver) de 0.25 a 0.40?
p_reg_mejorado = 0.40
p_caso_1_mejorado = p_ver * p_reg_mejorado
p_total_mejorado = p_caso_1_mejorado + p_caso_2
print(f'Si P(registro|ver) sube a {p_reg_mejorado}:')
print(f'  P(registro por video): {p_caso_1_mejorado:.3f}')
print(f'  P(registro total):     {p_total_mejorado:.3f}')

print(f'\nRecomendación de negocio: {recomendacion_negocio}')

---
## PARTE 4 — Actividad autónoma: App educativa (slides 24-27)

### Código exacto del slide 27

In [ ]:
# Código exacto de la presentación — slide 27
# NOTA: en el slide 27 aparece p_abrir = 0.5 pero el enunciado (slide 25) dice 0.55
# Se respeta el código del slide 27 exactamente
p_abrir = 0.5
p_inicia_despues_abrir = 0.35
p_no_abrir = 1 - p_abrir
p_segunda_notif = 0.5
p_inicia_despues_segunda = 0.15

p_caso1 = p_abrir * p_inicia_despues_abrir
p_caso2 = p_no_abrir * p_segunda_notif * p_inicia_despues_segunda
p_total = p_caso1 + p_caso2

print(f'Inicio tras abrir 1ª notif: {p_caso1:.2f}')
print(f'Inicio tras segunda notif: {p_caso2:.2f}')
print(f'Probabilidad total de inicio: {p_total:.2f}')

In [ ]:
# Construcción completa del árbol — paso 1 (slide 26)
print('=== ÁRBOL DE PROBABILIDADES — App educativa ===')
print()
print('Espacio muestral de la 1ª etapa:')
print('  S1 = {abre_notificacion, no_abre_notificacion}')
print(f'  P(abre) = {p_abrir}  |  P(no abre) = {p_no_abrir}')
print()
print('Etapa 2 (si abre):')
print(f'  P(inicia|abre) = {p_inicia_despues_abrir}')
print(f'  P(no inicia|abre) = {1-p_inicia_despues_abrir}')
print()
print('Etapa 2 (si no abre) → 2ª notificación:')
print(f'  P(recibe 2ª|no abre) = {p_segunda_notif}')
print(f'  P(inicia|recibe 2ª) = {p_inicia_despues_segunda}')
print()

# Tabla completa de caminos
caminos = pd.DataFrame({
    'Camino': [
        'Abre → Inicia',
        'Abre → No inicia',
        'No abre → 2ª notif → Inicia',
        'No abre → 2ª notif → No inicia',
        'No abre → Sin 2ª notif'
    ],
    'Cálculo': [
        f'{p_abrir}×{p_inicia_despues_abrir}',
        f'{p_abrir}×{1-p_inicia_despues_abrir}',
        f'{p_no_abrir}×{p_segunda_notif}×{p_inicia_despues_segunda}',
        f'{p_no_abrir}×{p_segunda_notif}×{1-p_inicia_despues_segunda}',
        f'{p_no_abrir}×{1-p_segunda_notif}'
    ],
    'Probabilidad': [
        round(p_caso1, 4),
        round(p_abrir*(1-p_inicia_despues_abrir), 4),
        round(p_caso2, 4),
        round(p_no_abrir*p_segunda_notif*(1-p_inicia_despues_segunda), 4),
        round(p_no_abrir*(1-p_segunda_notif), 4)
    ]
})
print(caminos.to_string(index=False))
print(f"\nSuma total: {caminos['Probabilidad'].sum():.4f}  ← debe ser 1.0 ✅")

In [ ]:
# Interpretación — paso 3 (slide 27)
print('=== INTERPRETACIÓN DE RESULTADOS (slide 27) ===')
print()
print(f'Camino 1 (abre 1ª notif + inicia): {p_caso1:.4f} = {p_caso1*100:.1f}%')
print(f'Camino 2 (2ª notif + inicia):       {p_caso2:.4f} = {p_caso2*100:.1f}%')
print(f'P(inicio total):                    {p_total:.4f} = {p_total*100:.1f}%')
print()

# ✏️ Completa la interpretación y recomendación
via_efectiva    = 'Camino 1' if p_caso1 > p_caso2 else 'Camino 2'
p_total_alta    = ""   # ✏️ ¿Es alta o baja la probabilidad total?
recomendacion   = ""   # ✏️ ¿Qué recomendarías al equipo?

print(f'Vía más efectiva:    {via_efectiva} (p={max(p_caso1,p_caso2):.4f})')
print(f'¿P total alta/baja?: {p_total_alta}')
print(f'Recomendación:       {recomendacion}')

### ✏️ Preguntas de cierre — slide 31:

In [ ]:
p1 = ""  # ¿Qué representa cada rama y nivel en el árbol?
p2 = ""  # ¿Cómo se calcula la probabilidad de una combinación secuencial?
p3 = ""  # ¿Cuándo usar unión y cuándo intersección?
p4 = ""  # ¿Mutuamente excluyentes vs independientes?
p5 = ""  # ¿Ventajas de Python para probabilidad?

preguntas = [
    '¿Qué representa cada rama y cada nivel en un árbol de probabilidad?',
    '¿Cómo se calcula la probabilidad de una combinación de eventos secuenciales?',
    '¿En qué casos se utiliza la regla de la unión y en cuáles la de la intersección?',
    '¿Qué significa que dos eventos sean mutuamente excluyentes? ¿Y que sean independientes?',
    '¿Qué ventajas ofrece usar Python para resolver problemas de probabilidad en contextos reales?'
]

print('--- PREGUNTAS DE CIERRE (slide 31) ---')
for i, (preg, resp) in enumerate(zip(preguntas, [p1,p2,p3,p4,p5]), 1):
    print(f'\n{i}. {preg}')
    print(f'   R: {resp}')

---
## 📋 Resumen

### Árbol de probabilidades

| Regla | Descripción | Fórmula |
|-------|-------------|--------|
| Multiplicación (rama) | P de una secuencia | `P(A) × P(B\|A)` |
| Suma de ramas hermanas | Siempre suman 1 | `P(A) + P(¬A) = 1` |
| Probabilidad total | Suma de todos los caminos al evento | `Σ P(camino_i)` |

### Unión e intersección

| Operación | Fórmula | Caso especial |
|-----------|---------|---------------|
| Unión P(A ∪ B) | P(A) + P(B) − P(A ∩ B) | Si disjuntos: P(A) + P(B) |
| Intersección independiente | P(A) × P(B) | Cuando P(B\|A) = P(B) |
| Intersección dependiente | P(A) × P(B\|A) | La regla general |
| Complemento | 1 − P(A) | Siempre válido |

### Relaciones entre eventos

| Relación | Test | Implicación |
|----------|------|-------------|
| Independientes | P(A∩B) = P(A)×P(B) | Uno no afecta al otro |
| Mutuamente excluyentes | P(A∩B) = 0 | No pueden coexistir |
| Inclusión A ⊂ B | P(A) ≤ P(B) | A ocurrir implica B ocurrir |

> 💡 **Truco:** cuando los caminos hacia un evento de interés son mutuamente excluyentes (como en los árboles), la probabilidad total es la SUMA de las probabilidades de cada camino.

> 💡 **Verificación siempre:** la suma de todas las hojas de un árbol debe ser exactamente 1.0. Si no lo es, hay un error en las probabilidades asignadas.